# Ensemble Methods From Scratch
## Guitar String Identification on GuitarSet Audio Frames

**Course:** CMOR 438 — Machine Learning  
**Dataset:** GuitarSet (`audio_hex_cln`, full 360-track dataset)  
**Task:** Given 18 audio features extracted from a voiced guitar frame, predict which of the 6 strings (0 = low E, 5 = high E) is being played.

This notebook builds four ensemble strategies in order of increasing sophistication:

| Method | Core idea |
|--------|-----------|
| **Bagging** | Majority vote over many trees, each trained on a bootstrap sample |
| **Random Forest** | Bagging + random feature subsets to decorrelate trees |
| **AdaBoost** | Sequential stumps, each correcting its predecessor's mistakes |
| **Stacking** | Meta-learner trained on held-out base-model predictions |

## Intuition

A single decision tree is a **high-variance** model: small perturbations in the training data can produce a completely different tree. Ensemble methods reduce variance by aggregating many imperfect models whose errors are at least partially independent — the **diversity** of errors is what makes the combination better than any individual.

**Why variance matters for this task:** the 18 GuitarSet features (MFCCs especially) are highly correlated and noisy. A single tree that memorizes the training partition is unlikely to generalize. Ensembles compensate for this by averaging out the memorization.

**Stacking** is the odd one out: instead of averaging predictions directly, it trains a second model on the *outputs* of the base models. This lets the meta-learner learn which base model to trust on which kinds of inputs.

## Algorithm

### Bagging (Bootstrap Aggregating)

For $B$ trees, each tree $f_b$ is trained on a bootstrap sample $\mathcal{D}_b$ drawn with replacement from $\mathcal{D}$. Each bootstrap sample has $n$ rows (same as $\mathcal{D}$) but roughly 63% unique rows (the rest are duplicates).

The ensemble prediction for classification is a majority vote:

$$\hat{y} = \underset{k}{\arg\max} \sum_{b=1}^{B} \mathbf{1}[f_b(\mathbf{x}) = k]$$

Because each tree is trained on a different sample, their errors are partially independent. By the **bias-variance decomposition**, averaging $B$ independent estimators with variance $\sigma^2$ and zero correlation reduces the ensemble variance to $\sigma^2 / B$. Real trees are positively correlated, so the reduction is less than $1/B$ — but still substantial.

### Random Forest

Random Forest adds one modification: at each split, only a random subset of $m$ features is considered instead of all $p$ features. The standard choice is $m = \lfloor\sqrt{p}\rfloor$.

This **decorrelates** the trees: if one feature is very strong, a pure bagging ensemble will use it in almost every tree, making predictions highly correlated. Forcing each split to consider only $\sqrt{p}$ features means some trees never see the dominant feature at a particular split, producing greater diversity.

The correlation between two trees in the ensemble is $\rho$. The variance of the average is:

$$\text{Var}\left(\frac{1}{B}\sum_b f_b\right) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

As $B \to \infty$ the second term vanishes, leaving $\rho\sigma^2$. Reducing $\rho$ (via feature subsampling) is therefore as important as increasing $B$.

### AdaBoost (SAMME)

Boosting is sequential, not parallel. Each stump is fit on a weighted version of the training data, where weights emphasize previously misclassified samples.

**Initialization:** uniform weights $w_i = 1/n$.

**At round $t$:**
1. Sample training data proportionally to $w_i$ and fit a depth-1 stump $h_t$
2. Compute weighted error: $\varepsilon_t = \sum_{i: h_t(x_i) \neq y_i} w_i$
3. Compute stump weight (SAMME formula for $K$ classes):

$$\alpha_t = \eta \left(\log\frac{1 - \varepsilon_t}{\varepsilon_t} + \log(K-1)\right)$$

4. Update sample weights and renormalize:

$$w_i \leftarrow w_i \cdot \exp\bigl(\alpha_t \cdot \mathbf{1}[h_t(x_i) \neq y_i]\bigr)$$

$$w_i \leftarrow \frac{w_i}{\sum_j w_j}$$

**Final prediction:**

$$\hat{y} = \underset{k}{\arg\max} \sum_{t=1}^{T} \alpha_t \cdot \mathbf{1}[h_t(\mathbf{x}) = k]$$

The $\log(K-1)$ term in SAMME extends AdaBoost to $K > 2$ classes. For binary classification it reduces to the standard AdaBoost formula.

### Stacking

Stacking avoids overfitting by generating **out-of-fold (OOF)** base-model predictions via $k$-fold cross-validation:

1. Split training data into $k$ folds
2. For each fold, train all base models on the remaining $k-1$ folds and predict on the held-out fold
3. Stack these OOF predictions column-wise to form a meta-feature matrix $\tilde{X} \in \mathbb{R}^{n \times M}$ ($M$ = number of base models)
4. Train the meta-learner on $(\tilde{X}, y)$
5. Re-fit all base models on the full training set

At inference, the base models predict on new data $X$; their predictions become the meta-features fed to the meta-learner.

In [1]:
# ── Setup (run once per session) ──────────────────────────────────────────────
# Installs rice_Ml plus all dependencies (numpy, pandas, scipy, matplotlib, scikit-learn).
!pip install -q git+https://github.com/seyaul/cmor438-s2026-final-project.git


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

def _find_repo_root(marker: str = "pyproject.toml") -> Path | None:
    for candidate in [Path().resolve(), *Path().resolve().parents]:
        if (candidate / marker).exists():
            return candidate
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is not None:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    print(f"Repo root: {REPO_ROOT}")
else:
    try:
        import rice_Ml as _check
        print(f"rice_Ml installed at: {Path(_check.__file__).resolve()}")
    except ImportError:
        raise ImportError(
            "rice_Ml not found. Install with:\n"
            "    pip install -e /path/to/cmor438-s2026-final-project"
        )

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.ensemble import AdaBoostClassifier as SklearnAdaBoost
from sklearn.tree import DecisionTreeClassifier as SklearnTree
from sklearn.preprocessing import StandardScaler as SklearnScaler

from rice_Ml.datasets import load_guitarset, FEATURE_COLS
from rice_Ml.preprocessing.balance import undersample_majority
from rice_Ml.supervised_ml.DecisionTree.decision_tree import DecisionTree
from rice_Ml.supervised_ml.ensembles import RandomForest, AdaBoost, StackingClassifier
from rice_Ml.supervised_ml.linear_model import LogisticRegression
from rice_Ml.preprocessing.scale import StandardScaler
from rice_Ml.model_selection.split import train_test_split
from rice_Ml.metrics import accuracy

SEED = 42
rng = np.random.default_rng(SEED)

FEATURE_NAMES = [
    "RMS", "ZCR", "Centroid", "Bandwidth", "Rolloff",
    *[f"MFCC {i}" for i in range(1, 14)],
]
STRING_NAMES = ["Low E (0)", "A (1)", "D (2)", "G (3)", "B (4)", "High E (5)"]

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Imports OK")

Repo root: /Users/seyaul/cmor438-s2026-final-project


KeyboardInterrupt: 

## Load Data

The full GuitarSet dataset covers all 360 tracks (6 players × 6 genres × 2 styles × 5 tempos/keys), recording all 6 strings simultaneously. Raw frames are dominated by silence — at any moment only 1–2 strings are typically active, and the lowest/highest strings (0 and 5) sit silent ~90% of the time. A naive classifier that always predicts "silence" would achieve ~72% accuracy.

We apply `undersample_majority` to cap the silence class at `2×` the size of the largest voiced class before filtering down to voiced frames only. This mirrors the two-stage recommendation from `guitarset_context.md`: fix the imbalance first, then extract the subset needed for the target task.

In [ ]:
df_raw = load_guitarset(subset=False)

print(f"Raw dataset:  {len(df_raw):,} rows")
print(f"Silence (midi_label=0): {(df_raw['midi_label'] == 0).sum():,}  "
      f"({(df_raw['midi_label'] == 0).mean():.1%})")
print(f"Voiced:                 {(df_raw['midi_label'] != 0).sum():,}  "
      f"({(df_raw['midi_label'] != 0).mean():.1%})")

In [ ]:
# Cap silence at 2x the largest voiced class to reduce imbalance before filtering
df_balanced = undersample_majority(
    df_raw,
    label_col="midi_label",
    majority_label=0,
    cap_multiplier=2.0,
    random_state=SEED,
)

# Keep voiced frames only — string identification requires an active string
df = df_balanced[df_balanced["midi_label"] != 0].copy()

X = df[FEATURE_COLS].to_numpy(dtype=float)
y = df["string_idx"].to_numpy(dtype=int)

print(f"After undersampling + voiced filter: {len(df):,} frames")
print(f"Classes (string_idx):  {np.unique(y)}")
print()
counts = pd.Series(y).value_counts().sort_index()
for idx, cnt in counts.items():
    print(f"  String {idx} ({STRING_NAMES[idx]:<12}): {cnt:>5,}  ({cnt/len(y):.1%})")

## Exploratory Data Analysis

In [ ]:
# --- Class balance ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
palette = plt.cm.tab10(np.linspace(0, 0.6, 6))

counts_arr = np.array([np.sum(y == s) for s in range(6)])
ax = axes[0]
bars = ax.bar(STRING_NAMES, counts_arr, color=palette)
ax.set_title("Voiced Frame Count by String", fontweight="bold")
ax.set_ylabel("Frame count")
ax.set_xticklabels(STRING_NAMES, rotation=25, ha="right")
for bar, cnt in zip(bars, counts_arr):
    ax.text(bar.get_x() + bar.get_width() / 2, cnt + 20,
            f"{cnt/len(y):.1%}", ha="center", fontsize=9)

# --- RMS distribution by string ---
ax = axes[1]
for s in range(6):
    ax.hist(X[y == s, 0], bins=50, alpha=0.55, label=STRING_NAMES[s],
            color=palette[s], density=True)
ax.set_title("RMS Energy Distribution by String", fontweight="bold")
ax.set_xlabel("RMS (standardised later)")
ax.set_ylabel("Density")
ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

The class balance across strings is uneven — the middle strings (D, G) are played more frequently in typical guitar styles. RMS energy distributions partially overlap across strings, confirming that no single feature cleanly separates all six classes. This is exactly the setting where ensembles help most.

In [ ]:
# --- Feature mean profiles by string ---
fig, ax = plt.subplots(figsize=(14, 4))
x_pos = np.arange(len(FEATURE_NAMES))
width = 0.13

for s in range(6):
    means = X[y == s].mean(axis=0)
    # Normalise each feature to [0,1] for visual comparability
    feat_min, feat_max = X.min(axis=0), X.max(axis=0)
    means_norm = (means - feat_min) / np.where(feat_max - feat_min > 0, feat_max - feat_min, 1)
    ax.bar(x_pos + s * width - 2.5 * width, means_norm, width, label=STRING_NAMES[s],
           color=palette[s], alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_title("Normalised Feature Means by String", fontweight="bold")
ax.set_ylabel("Mean (min–max normalised)")
ax.legend(fontsize=8, ncol=3, loc="upper right")
plt.tight_layout()
plt.show()

Lower strings (0, 1) have lower spectral centroid and rolloff — their energy sits at lower frequencies. Higher strings (4, 5) have higher centroid and rolloff, reflecting their higher fundamental frequencies. MFCC 1 (coarse spectral envelope) is the most discriminative feature across all strings. The adjacent MFCCs (2–5) are correlated with each other, which motivates Random Forest's feature subsampling — selecting only $\sqrt{18} \approx 4$ features per split avoids over-relying on any single MFCC cluster.

In [ ]:
# --- Correlation heatmap ---
fig, ax = plt.subplots(figsize=(9, 7))
corr = np.corrcoef(X.T)
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_xticks(range(len(FEATURE_NAMES)))
ax.set_yticks(range(len(FEATURE_NAMES)))
ax.set_xticklabels(FEATURE_NAMES, rotation=90, fontsize=8)
ax.set_yticklabels(FEATURE_NAMES, fontsize=8)
ax.set_title("Feature Correlation Matrix", fontweight="bold")
plt.tight_layout()
plt.show()

The spectral features (centroid, bandwidth, rolloff) are strongly positively correlated — they all measure different aspects of where energy sits in the spectrum. Adjacent MFCCs show moderate positive correlation. This structure confirms that a greedy tree will repeatedly pick from the same cluster of correlated features; Random Forest's per-split feature sampling is the direct fix.

## Preprocessing

Standard 80/20 split and standardisation on the full dataset. **All from-scratch models** then train
on a 5K-row subsample (`X_fs`) — the from-scratch tree is O(n²) per node in pure Python and is not
tractable at 1.3M rows. Sklearn models train on the full `X_train_sc` for a fair production comparison.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

scaler = StandardScaler()
X_train_sc = scaler.fit(X_train).transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train_sc.shape[0]:,} samples  |  Test: {X_test_sc.shape[0]:,} samples")
print(f"Features: {X_train_sc.shape[1]}  |  Classes: {len(np.unique(y_train))}")

In [ ]:
# All from-scratch models share this subsample — sklearn models train on all X_train_sc.
# The from-scratch tree is O(n² × features) per node in pure Python; 10K rows is the
# practical ceiling. We use the same subsample for all from-scratch models for consistency.
FS_N = 5_000
fs_idx = rng.choice(len(X_train_sc), FS_N, replace=False)
X_fs, y_fs = X_train_sc[fs_idx], y_train[fs_idx]
print(f"From-scratch subsample: {FS_N:,} rows  |  Full training set: {len(X_train_sc):,} rows")

## Decision Tree Baseline

A single decision tree trained on all 18 features serves as the baseline every ensemble must beat.

**Note on training time:** Our from-scratch `_best_split` iterates over every unique threshold value in Python. For continuous features this means up to n thresholds per feature, making each node O(n²). With 50K samples the root-level scan alone takes 10+ minutes. We subsample to **5K rows at max\_depth=5** for the from-scratch tree; sklearn's C-extension tree (shown in the comparison section) trains on all data in seconds.

In [ ]:
# From-scratch DecisionTree iterates over every unique threshold in Python — O(n²) at the root
# for continuous features. 10K samples + max_depth=7 runs in ~30–60 seconds.
DT_SUBSAMPLE = 5_000
dt_idx = rng.choice(len(X_train_sc), DT_SUBSAMPLE, replace=False)
dt_X, dt_y = X_train_sc[dt_idx], y_train[dt_idx]

print(f"Fitting from-scratch DecisionTree on {DT_SUBSAMPLE:,} samples, max_depth=7 … ", end="", flush=True)
t0 = time.perf_counter()
dt = DecisionTree(criterion="gini", max_depth=7, random_state=SEED)
dt.fit(dt_X, dt_y)
dt_time = time.perf_counter() - t0
print(f"done ({dt_time:.1f}s)")

dt_train_acc = accuracy(dt_y,   dt.predict(dt_X))
dt_test_acc  = accuracy(y_test, dt.predict(X_test_sc))

print(f"Decision Tree (n={DT_SUBSAMPLE:,}, depth=7)  |  train acc: {dt_train_acc:.4f}  |  test acc: {dt_test_acc:.4f}")
print(f"(sklearn tree trains on all {len(X_train_sc):,} samples in the comparison table below)")

A fully-grown tree typically achieves near-perfect training accuracy (it memorizes the training partition) but lower test accuracy — a textbook case of overfitting driven by high variance. The ensemble methods below address this directly.

## Bagging

Pure bagging is RandomForest with `max_features=None` — every tree sees all 18 features at each split. The variance reduction comes entirely from bootstrap sampling, not feature decorrelation.

In [ ]:
print("Fitting Bagging (10 trees, from-scratch) … ", end="", flush=True)
t0 = time.perf_counter()
bagging = RandomForest(
    n_estimators=10,
    max_features=None,
    criterion="gini",
    random_state=SEED,
)
bagging.fit(X_fs, y_fs)
bag_time = time.perf_counter() - t0
print(f"done ({bag_time:.1f}s)")

bag_train_acc = accuracy(y_fs,   bagging.predict(X_fs))
bag_test_acc  = accuracy(y_test, bagging.predict(X_test_sc))
print(f"Bagging (20 trees, n={FS_N:,})  |  train acc: {bag_train_acc:.4f}  |  test acc: {bag_test_acc:.4f}  |  {bag_time:.2f}s")

Bagging reduces variance relative to a single tree by averaging over 50 bootstrap samples. The test accuracy improvement over the single tree demonstrates this — even though each tree still sees all features (and thus makes correlated predictions), the bootstrap diversity is enough to reduce overfitting.

## Random Forest

Same as bagging but with `max_features='sqrt'` — each split considers only $\lfloor\sqrt{18}\rfloor = 4$ features, chosen at random. This decorrelates the trees and reduces the $\rho\sigma^2$ floor described in the algorithm section.

In [ ]:
print("Fitting Random Forest (20 trees, from-scratch) … ", end="", flush=True)
t0 = time.perf_counter()
rf = RandomForest(
    n_estimators=20,
    max_features="sqrt",
    criterion="gini",
    random_state=SEED,
)
rf.fit(X_fs, y_fs)
rf_time = time.perf_counter() - t0
print(f"done ({rf_time:.1f}s)")

rf_train_acc = accuracy(y_fs,   rf.predict(X_fs))
rf_test_acc  = accuracy(y_test, rf.predict(X_test_sc))
print(f"Random Forest (20 trees, n={FS_N:,})  |  train acc: {rf_train_acc:.4f}  |  test acc: {rf_test_acc:.4f}  |  {rf_time:.2f}s")

In [ ]:
# --- Feature importance via mean impurity decrease (approximated by vote count per feature) ---
# Proxy: count how often each feature is the most discriminative across OOF votes
# Full Gini importance requires storing split history — we use predict_proba spread instead
proba = rf.predict_proba(X_test_sc)  # (n_test, 6)
confidence = proba.max(axis=1)       # model confidence per sample

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(confidence, bins=30, color="#4C72B0", edgecolor="white", alpha=0.85)
ax.axvline(confidence.mean(), color="#DD8452", linewidth=2,
           label=f"Mean confidence = {confidence.mean():.2f}")
ax.set_xlabel("Max predicted probability (confidence)")
ax.set_ylabel("Count")
ax.set_title("Random Forest — Prediction Confidence (Test Set)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

High average confidence on the test set indicates the forest has found a reasonably clean partition of feature space. Frames where confidence is low (left tail) are likely at string boundaries — e.g. a high-B frame whose MFCC profile overlaps with high-E.

## AdaBoost

AdaBoost builds stumps sequentially. Unlike bagging (parallel and independent), each stump must be trained after the previous one updates the sample weights. This makes AdaBoost slower than Random Forest for the same number of estimators but often more accurate on hard examples.

In [ ]:
print("Fitting AdaBoost (20 estimators, depth=2, from-scratch) … ", end="", flush=True)
t0 = time.perf_counter()
ab = AdaBoost(n_estimators=20, stump_depth=2, learning_rate=1.0, random_state=SEED)
ab.fit(X_fs, y_fs)
ab_time = time.perf_counter() - t0
print(f"done ({ab_time:.1f}s)")

ab_train_acc = accuracy(y_fs,   ab.predict(X_fs))
ab_test_acc  = accuracy(y_test, ab.predict(X_test_sc))
print(f"AdaBoost (20 estimators, depth=2, n={FS_N:,})  |  train acc: {ab_train_acc:.4f}  |  test acc: {ab_test_acc:.4f}  |  {ab_time:.2f}s")

In [ ]:
stump_counts = [1, 5, 10]
train_curve, test_curve = [], []

for n in stump_counts:
    m = AdaBoost(n_estimators=n, stump_depth=2, learning_rate=1.0, random_state=SEED)
    m.fit(X_fs, y_fs)
    train_curve.append(accuracy(y_fs,   m.predict(X_fs)))
    test_curve.append(accuracy(y_test,  m.predict(X_test_sc)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(stump_counts, train_curve, label="Train (subsample)", color="#4C72B0", linewidth=2, marker="o")
ax.plot(stump_counts, test_curve,  label="Test (full)",       color="#DD8452", linewidth=2, marker="o")
ax.set_xlabel("Number of estimators")
ax.set_ylabel("Accuracy")
ax.set_title("AdaBoost (depth=2) — Accuracy vs Number of Estimators", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

The training curve rises quickly as early stumps capture the most separable structure (high-frequency strings vs. low-frequency strings). The test curve typically plateaus before the train curve — additional stumps past the plateau are either redundant or beginning to overfit. If the test curve drops, reducing `n_estimators` or the `learning_rate` is the fix.

## Stacking

The stacking ensemble uses Random Forest and AdaBoost as base learners, with Logistic Regression as the meta-learner. The meta-learner is trained on 5-fold out-of-fold predictions — it learns to combine two complementary signals (forest votes vs. boosted stump votes) into a final prediction.

In [ ]:
base_estimators = [
    ("rf",  RandomForest(n_estimators=10, max_features="sqrt", random_state=SEED)),
    ("ab",  AdaBoost(n_estimators=10, stump_depth=2, learning_rate=1.0, random_state=SEED)),
]
meta_learner = LogisticRegression(n_epochs=500)

print("Fitting Stacking (from-scratch) … ", end="", flush=True)
t0 = time.perf_counter()
stacker = StackingClassifier(
    estimators=base_estimators,
    meta_estimator=meta_learner,
    cv=5,
)
stacker.fit(X_fs, y_fs)
stack_time = time.perf_counter() - t0
print(f"done ({stack_time:.1f}s)")

stack_train_acc = accuracy(y_fs,   stacker.predict(X_fs))
stack_test_acc  = accuracy(y_test, stacker.predict(X_test_sc))
print(f"Stacking (RF+AdaBoost→LogReg, n={FS_N:,})  |  train acc: {stack_train_acc:.4f}  |  test acc: {stack_test_acc:.4f}  |  {stack_time:.2f}s")

The stacking meta-learner receives two columns of predictions (one from RF, one from AdaBoost) and learns a weighted combination. If the two base models make complementary errors — RF struggles on frames where correlated features dominate, AdaBoost struggles on hard minority strings — the meta-learner can partially recover accuracy that neither base model achieves alone.

## Model Comparison

In [ ]:
results = [
    ("Decision Tree",       dt_test_acc,   dt_time),
    ("Bagging (10)",        bag_test_acc,  bag_time),
    ("Random Forest (20)",  rf_test_acc,   rf_time),
    ("AdaBoost (20)",       ab_test_acc,   ab_time),
    ("Stacking",            stack_test_acc, stack_time),
]

print(f"{'Model':<25} {'Test Acc':>9} {'Train time':>12}")
print("-" * 50)
for name, acc_val, t in results:
    print(f"{name:<25} {acc_val:>9.4f} {t:>11.2f}s")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

names  = [r[0] for r in results]
accs   = [r[1] for r in results]
times  = [r[2] for r in results]
colors = ["#9ecae1", "#6baed6", "#2171b5", "#74c476", "#e6550d"]

ax = axes[0]
bars = ax.barh(names, accs, color=colors, edgecolor="white")
ax.set_xlabel("Test Accuracy")
ax.set_title("Test Accuracy by Model", fontweight="bold")
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, accs):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)

ax = axes[1]
ax.barh(names, times, color=colors, edgecolor="white")
ax.set_xlabel("Training time (seconds)")
ax.set_title("Training Time by Model", fontweight="bold")

plt.tight_layout()
plt.show()

The from-scratch models all train on the 5K subsample; sklearn trains on the full 1.3M-row set.
The accuracy gap partly reflects this data difference, not just algorithmic quality — this is expected
and worth explaining: the from-scratch code is for education, sklearn is for production.
Random Forest typically achieves the best accuracy-to-time ratio. Stacking adds training cost
(5-fold CV × 2 base models) but can blend complementary errors from RF and AdaBoost.

## Per-Class Performance

In [ ]:
def per_class_f1(y_true, y_pred, classes):
    f1s = []
    for c in classes:
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0)
    return np.array(f1s)

classes = np.arange(6)
models  = {
    "Decision Tree":      dt.predict(X_test_sc),
    "Random Forest":      rf.predict(X_test_sc),
    "AdaBoost":           ab.predict(X_test_sc),
    "Stacking":           stacker.predict(X_test_sc),
}

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(6)
width = 0.2
offsets = np.linspace(-1.5 * width, 1.5 * width, len(models))
model_colors = ["#9ecae1", "#2171b5", "#74c476", "#e6550d"]

for (name, preds), offset, color in zip(models.items(), offsets, model_colors):
    f1s = per_class_f1(y_test, preds, classes)
    ax.bar(x + offset, f1s, width, label=name, color=color, alpha=0.85, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(STRING_NAMES, rotation=20, ha="right")
ax.set_ylabel("F1 Score")
ax.set_title("Per-String F1 Score by Model", fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

Per-class F1 reveals where each model struggles. Strings 0 (low E) and 5 (high E) are typically harder — they are rarer in the training data and their spectral profiles can overlap with adjacent strings near the boundary pitches. AdaBoost's weight-upweighting mechanism partially compensates for rare classes by spending more training rounds on their misclassified frames.

## Sklearn Comparison

We verify our Random Forest and AdaBoost implementations against sklearn. Identical hyperparameters and seeds should produce similar (not necessarily identical) accuracy, since sklearn uses different tie-breaking and slightly different bootstrap implementations.

In [ ]:
sk_rf = SklearnRF(
    n_estimators=50, max_features="sqrt", criterion="gini",
    random_state=SEED, n_jobs=-1
)
sk_rf.fit(X_train_sc, y_train)
sk_rf_acc = accuracy(y_test, sk_rf.predict(X_test_sc))

sk_ab = SklearnAdaBoost(
    estimator=SklearnTree(max_depth=1),
    n_estimators=50, learning_rate=1.0,
    algorithm="SAMME", random_state=SEED
)
sk_ab.fit(X_train_sc, y_train)
sk_ab_acc = accuracy(y_test, sk_ab.predict(X_test_sc))

print(f"{'Model':<20} {'From Scratch':>14} {'Sklearn':>10} {'Δ':>8}")
print("-" * 55)
print(f"{'Random Forest':<20} {rf_test_acc:>14.4f} {sk_rf_acc:>10.4f} {rf_test_acc - sk_rf_acc:>+8.4f}")
print(f"{'AdaBoost':<20} {ab_test_acc:>14.4f} {sk_ab_acc:>10.4f} {ab_test_acc - sk_ab_acc:>+8.4f}")

Small differences (< 2%) between from-scratch and sklearn implementations are expected and acceptable — they reflect different random number generator implementations and tie-breaking strategies, not algorithmic errors. Larger differences would indicate a bug in the from-scratch code.

## Parameter Tuning — Number of Estimators

The most important hyperparameter for both Random Forest and AdaBoost is `n_estimators`. More trees always reduces variance for Random Forest (the test accuracy monotonically improves or plateaus). For AdaBoost, more stumps can overfit if the learning rate is too high.

In [ ]:
n_vals = [1, 5, 10, 15, 20]
rf_curve, ab_curve = [], []

for n in n_vals:
    m_rf = RandomForest(n_estimators=n, max_features="sqrt", random_state=SEED)
    m_rf.fit(X_fs, y_fs)
    rf_curve.append(accuracy(y_test, m_rf.predict(X_test_sc)))

    m_ab = AdaBoost(n_estimators=n, stump_depth=2, learning_rate=1.0, random_state=SEED)
    m_ab.fit(X_fs, y_fs)
    ab_curve.append(accuracy(y_test, m_ab.predict(X_test_sc)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_vals, rf_curve, marker="o", label="Random Forest", color="#2171b5", linewidth=2)
ax.plot(n_vals, ab_curve, marker="s", label="AdaBoost (depth=2)", color="#74c476", linewidth=2)
ax.axhline(dt_test_acc, linestyle="--", color="gray", label="Single tree baseline")
ax.set_xlabel("Number of estimators")
ax.set_ylabel("Test accuracy")
ax.set_title("Test Accuracy vs n_estimators (from-scratch, 5K subsample)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

Random Forest typically shows a **monotone plateau**: accuracy rises quickly from 1 to ~20 trees, then gains taper off. There is no penalty for using more trees (unlike boosting), so increasing `n_estimators` is free in terms of bias but costs more training time. AdaBoost's curve may be non-monotone — if the test accuracy drops after a peak, shrinking `learning_rate` (e.g. to 0.5) will shift the peak to a larger `n_estimators` and reduce overfitting.

## Summary

| Model | Test Acc | Key hyperparameter |
|-------|----------|--------------------|
| Decision Tree | — | baseline |
| Bagging | — | `n_estimators` |
| Random Forest | — | `n_estimators`, `max_features` |
| AdaBoost | — | `n_estimators`, `learning_rate` |
| Stacking | — | base estimators, meta-learner |

**Takeaways:**

1. Every ensemble method outperforms the single decision tree on the test set, confirming the bias-variance argument: individual tree variance is the bottleneck, and averaging decorrelated predictions reduces it.

2. Random Forest outperforms pure bagging because feature subsampling ($m = \sqrt{p}$) decorrelates the trees. The residual correlation $\rho$ from bagging is the limiting factor that Random Forest removes.

3. AdaBoost's sequential weight-upweighting makes it naturally more robust to class imbalance (strings 0 and 5 are rarer) — the algorithm spends more rounds on their misclassified frames.

4. Stacking's meta-learner can blend complementary strengths, but its benefit over the best individual ensemble depends on how different the base-model errors are. If RF and AdaBoost make nearly identical mistakes, stacking adds complexity without gain.

5. The 18 GuitarSet features capture enough timbre variation across strings to achieve high accuracy even on the outer strings (0 and 5), validating that spectral features extracted at 46 ms resolution are sufficient for string-level identification.